In [1]:
# 1. load + feature matrix
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df = pd.read_parquet("../data/constructs/slope_feature_table.parquet")
FEATS = ["elev","slope","aspect_sin","aspect_cos","roughness",
         "tri","tpi","ndvi","lst","dist"]
X = df[FEATS].to_numpy(np.float64)
print("rows:", X.shape[0], "| features:", X.shape[1],
      "| NaNs:", int(np.isnan(X).sum()))

rows: 2575353 | features: 10 | NaNs: 0


In [2]:
# Cell — binary label from global ddem stats
mu, sd = df["ddem"].mean(), df["ddem"].std()
lo, hi = mu - sd/2, mu + sd/2
df["moving"] = ((df["ddem"] < lo) | (df["ddem"] > hi)).astype(int)
print(f"stable band: [{lo:.2f}, {hi:.2f}] m")
print(df["moving"].value_counts(normalize=True).round(3))
print(df.groupby(["glacier","side"])["moving"].mean().round(3))

stable band: [-1.72, 1.50] m
moving
0    0.624
1    0.376
Name: proportion, dtype: float64
glacier  side
canwell  N       0.417
         S       0.418
castner  N       0.412
         S       0.366
fels     N       0.275
         S       0.477
Name: moving, dtype: float64


In [2]:
# build in 2010-2025 dDEM to compare with residual maps

import numpy as np, rasterio, pandas as pd
from rasterio.transform import rowcol

df = pd.read_parquet("../data/constructs/slope_feature_table.parquet")
can = df[df.slope_id.isin([1, 2])].copy()          # Canwell N(1) + S(2)
xs = can["x"].to_numpy(); ys = can["y"].to_numpy()

with rasterio.open("../data/rasters/Canwell_ddem_2025min2010_5m.tif") as s:   # your 2025−2010 dDEM (EPSG:32606, 5 m)
    fut = s.read(1); nod = s.nodata
    rr, cc = rowcol(s.transform, xs, ys)           # sample by map coords, robust to a different window
rr = np.asarray(rr); cc = np.asarray(cc)
ok = (rr >= 0) & (rr < fut.shape[0]) & (cc >= 0) & (cc < fut.shape[1])
fut_v = np.full(len(can), np.nan, "float32")
fut_v[ok] = fut[rr[ok], cc[ok]]
fut_v[fut_v == nod] = np.nan

FEATS = ["elev","slope","aspect_sin","aspect_cos","roughness","tri","tpi","ndvi","lst","dist"]
np.savez_compressed("../data/constructs/canwell_resid_check.npz",
    X=can[FEATS].to_numpy("float32"),
    ddem_0010=can["ddem"].to_numpy("float32"),     # 2000–2010 (ML target)
    ddem_1025=fut_v,                               # 2010–2025 (future)
    slope_id=can["slope_id"].to_numpy("int16"),
    x=xs, y=ys)
print("future valid:", int(np.isfinite(fut_v).sum()), "of", len(can))

future valid: 668137 of 731027
